In [1]:
import pandas as pd

In [2]:
retail= pd.read_csv("../data/retail.csv", index_col=0)

In [3]:
retail.info()

<class 'pandas.DataFrame'>
Index: 1400 entries, ORD-10000 to ORD-11399
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          1400 non-null   str    
 1   category      1400 non-null   str    
 2   product       1400 non-null   str    
 3   region        1400 non-null   str    
 4   channel       1400 non-null   str    
 5   units         1400 non-null   int64  
 6   unit_price    1400 non-null   float64
 7   discount_pct  1400 non-null   int64  
 8   revenue       1400 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 109.4+ KB


In [14]:
retail['category'].size

1400

In [17]:
retail.groupby('category').size()

category
Books          355
Electronics    364
Home           340
Stationery     341
dtype: int64

In [ ]:
# υποθετικό σενάριο: συμπλήρωσε 0 για τα κενά στην στήλη units
retail['units']= retail['units'].fillna(0)

In [6]:
# 1. Δημιούργησε μια νέα στήλη στον πίνακα retail με όνομα unit_price, η οποία θα υπολογίζει 
# την τιμή ανά τεμάχιο (revenue / units).
# 2. Φιλτράρισε τον πίνακα ώστε να κρατήσεις μόνο τις παραγγελίες που έγιναν είτε στην 
# περιοχή 'North' είτε στην περιοχή 'South', ΚΑΙ είχαν πάνω από 10 τεμάχια (units > 10)

retail['unit_price']= retail['revenue']/retail['units']

retail[
    (retail['region'].isin(['North','South']))&(retail['units']>10)
]

,date,category,product,region,channel,units,unit_price,discount_pct,revenue
order_id,,,,,,,,,


In [7]:
# Βρες τις 5 κορυφαίες παραγγελίες με τα μεγαλύτερα έσοδα (revenue) για την κατηγορία 'Electronics'.
# Το αποτέλεσμα πρέπει να δείχνει μόνο τις στήλες region, channel, και revenue, 
# ταξινομημένες από το μεγαλύτερο έσοδο προς το μικρότερο.

retail.loc[
    retail['category']=='Electronics',
    ['region','channel','revenue']
].sort_values('revenue', ascending=False).head(5)


,region,channel,revenue
order_id,,,
ORD-10728,West,app,14400.0
ORD-11139,East,app,12000.0
ORD-10350,South,web,12000.0
ORD-10472,West,app,12000.0
ORD-11008,North,store,12000.0


In [8]:
#Ομαδοποίησε τα δεδομένα ανά κανάλι πώλησης (channel) και υπολόγισε με τη χρήση της .agg():
# 	Το συνολικό πλήθος των παραγγελιών (total_orders), προσέχοντας να μην επηρεαστεί από τυχόν κενές τιμές (NaN).
# 	Τον μέσο όρο των εσόδων (avg_revenue).
#   Τη μέγιστη παραγγελία σε τεμάχια (max_units).

(
    retail.groupby('channel')
    .agg(total_orders=('channel','count'),
         avg_revenue=('revenue','mean'),
         max_units=('units','max')
         ).round()
         .reset_index()
         .sort_values(by=['avg_revenue'],ascending=False)
)

,channel,total_orders,avg_revenue,max_units
0,app,523,1819.0,7
1,store,360,1374.0,7
2,web,517,1143.0,8


In [9]:
# Χρησιμοποίησε τη μέθοδο .groupby() και την .filter() (ή having λογική) για να επιστρέψεις 
# τις γραμμές των δεδομένων μόνο για εκείνες τις περιοχές (region) των οποίων το 
# συνολικό άθροισμα εσόδων (revenue) ξεπερνά τα 500.000.

(
    retail.groupby('region')['revenue']
    .sum()
    .reset_index(name='sum')
    .query("sum>500000")
    .round()
    .sort_values('sum',ascending=False)   
)

,region,sum
3,West,534974.0
2,South,516035.0
0,East,501554.0


In [10]:
region_groups = retail.groupby('region')['revenue'].sum().reset_index(name='sum')
region_groups[region_groups['sum']>500000].sort_values('sum',ascending=False)

,region,sum
3,West,534974.45
2,South,516035.35
0,East,501554.35


In [11]:
#Φτιάξε ένα νέο, συνοπτικό DataFrame (1 γραμμή, 3 στήλες) χρησιμοποιώντας Boolean Masks 
#(.sum() και .mean()), το οποίο θα δείχνει:
# 	1. app_orders: Πόσες παραγγελίες έγιναν μέσω 'app'.
# 	2. high_value_home: Πόσες παραγγελίες της κατηγορίας 'Home' έφεραν έσοδα πάνω από 500.
# 	3. web_percentage: Το ποσοστό (%) των συνολικών παραγγελιών που έγιναν μέσω 'web'.

dic= {
    'app_orders': (retail['channel']=='app').sum(),
    'high_value_home': ((retail['category']=='Home')&(retail['revenue']>500)).sum(),
    'web_percentage_%': ((retail['channel']=='web').mean()*100).round(2)
    }
new_df= pd.DataFrame(data=dic, index=[0])
new_df


,app_orders,high_value_home,web_percentage_%
0,523,295,36.93


In [12]:
#Ομαδοποίησε τα δεδομένα ανά κανάλι πώλησης (channel) και υπολόγισε με τη χρήση της .agg()
# για την στήλη revenue, το avg, median, SD και skew  

(
    retail.groupby('channel')['revenue']
    .aggregate(avg_rev='mean',median_rev='median',SD='std',skewness='skew') #ή .aggregate(['mean','median','std','skew'])
    .round(2)
    .reset_index()
)

,channel,avg_rev,median_rev,SD,skewness
0,app,1819.28,1050.0,2175.46,2.22
1,store,1374.18,750.0,1759.97,2.45
2,web,1142.60,700.0,1496.07,3.03
